In [2]:
!pip install -q torch transformers datasets accelerate
!pip install -q trl==0.8.0 # O 0.8.6 0.17.0
# !pip install -q trl # Para la última versión estable

# Verific. versiones instaladas
!pip show trl
!pip show transformers
!pip show accelerate
!pip show torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 107.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 12.4 MB/s eta 0:00:00
Name: trl
Version: 0.8.0
Summary: Train 

In [4]:
# -*- coding: utf-8 -*-
"""rlhf_trl_0_8_0_basic_generate_output_shape_fix.ipynb"""

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig
from trl import PPOConfig, PPOTrainer, AutoModelForCausalLMWithValueHead, create_reference_model
from datasets import Dataset
import os

# wandb (Weights & Biases)  https://wandb.ai/site
os.environ["WANDB_DISABLED"] = "true"

model_name = "gpt2"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLMWithValueHead.from_pretrained(model_name).to(device)
ref_model = create_reference_model(model).to(device)

generation_config = GenerationConfig(
    max_new_tokens=30,
    min_length=-1,
    do_sample=True,
    temperature=0.7,
    top_k=0.0,
    top_p=1.0,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

def reward_fn(texts_list):
    rewards = []
    for text in texts_list:
        if "hello" in text.lower():
            rewards.append(1.0)
        else:
            rewards.append(-1.0)
    return rewards

config = PPOConfig(
    model_name=model_name, learning_rate=1.41e-5, batch_size=1,
    mini_batch_size=1, gradient_accumulation_steps=1, ppo_epochs=4,
)

prompts_text = ["Say something friendly:", "Introduce yourself:"]
dataset_dict = {"query": prompts_text}
dataset = Dataset.from_dict(dataset_dict)

ppo_trainer = PPOTrainer(
    config=config, model=model, ref_model=ref_model, tokenizer=tokenizer,
    dataset=dataset,
)

num_ppo_steps = len(dataset)
print(f"Iniciando entrenamiento PPO para {num_ppo_steps} pasos...")
dataloader_iterator = iter(ppo_trainer.dataloader)

for ppo_step in range(num_ppo_steps):
    print(f"\n--- PPO Step {ppo_step + 1}/{num_ppo_steps} ---")

    try:
        batch = next(dataloader_iterator)
    except StopIteration:
        print("Dataloader agotado inesperadamente. Terminando el bucle.")
        break

    query_texts = batch["query"]
    tokenized_queries = tokenizer(query_texts, return_tensors="pt", padding=True, truncation=True, max_length=32)
    query_tensors_2d_batched = tokenized_queries["input_ids"].to(device)
    query_attention_mask_2d_batched = tokenized_queries.get("attention_mask", None)
    if query_attention_mask_2d_batched is not None:
        query_attention_mask_2d_batched = query_attention_mask_2d_batched.to(device)

    print(f"Query (texto): {query_texts[0]}")

    # Listas para los argumentos de ppo_trainer.step
    queries_for_step = []
    responses_generated_only_for_step = []
    generated_only_texts_for_reward = []

    # values puede tener shape:
    # 2D: como (batch_size, 1) → por ejemplo: [[0.54], [0.12], [0.77]]
    # 1D: como (batch_size,) → por ejemplo: [0.54, 0.12, 0.77]
    # se controla esto aqui porqe PyTorch da errores si se intenta hacer
    # operaciones como restas o promedios entre tensores de distinto shape.
    for i in range(query_tensors_2d_batched.shape[0]): # Iterar sobre el batch (tamaño 1)
        current_query_1d = query_tensors_2d_batched[i] # Tensor 1D del prompt [seq_len_prompt]
        queries_for_step.append(current_query_1d) # Guardar para step

        current_attn_mask_2d_for_generate = query_attention_mask_2d_batched[i:i+1] if query_attention_mask_2d_batched is not None else None # Forma [1, seq_len_prompt]

        gen_kwargs_single = generation_config.to_dict()
        if current_attn_mask_2d_for_generate is not None:
            gen_kwargs_single["attention_mask"] = current_attn_mask_2d_for_generate


        # Llamar a generate con el query 1D.
        # PPOTrainer.generate: Si query_tensor es 1D, devuelve 1D. Si es lista de 1D, devuelve lista de 1D.
        # Pero el error sugiere que el resultado del slicing (response_generated_only_1d) es 2D.
        # Esto implica que response_full (antes del slicing) era 2D [1, total_len].

        response_full_from_generate = ppo_trainer.generate(
            current_query_1d, # Se pasa 1D
            return_prompt=True,
            **gen_kwargs_single
        )
        # print(f"  Shape de response_full_from_generate: {response_full_from_generate.shape}")

        # Si response_full_from_generate es 1D [total_len] (como debería ser):
        if response_full_from_generate.ndim == 1:
            response_full_1d_for_item = response_full_from_generate
        # Si response_full_from_generate es 2D [1, total_len] (inesperado pero posible):
        elif response_full_from_generate.ndim == 2 and response_full_from_generate.shape[0] == 1:
            response_full_1d_for_item = response_full_from_generate.squeeze(0)
            print(f"  ADVERTENCIA: ppo_trainer.generate devolvió 2D, se hizo squeeze(0). Nueva forma: {response_full_1d_for_item.shape}")
        else:
            raise ValueError(f"Forma inesperada de ppo_trainer.generate: {response_full_from_generate.shape}")

        prompt_len = current_query_1d.shape[0]
        # response_generated_only_1d debe ser 1D
        response_generated_only_1d = response_full_1d_for_item[prompt_len:]
        # print(f"  Shape de response_generated_only_1d: {response_generated_only_1d.shape}")
        responses_generated_only_for_step.append(response_generated_only_1d)


        generated_text = tokenizer.decode(response_generated_only_1d, skip_special_tokens=True)
        generated_only_texts_for_reward.append(generated_text)

    print(f"Respuesta (solo generada): {generated_only_texts_for_reward[0]}")

    rewards = reward_fn(generated_only_texts_for_reward)
    rewards_pt = [torch.tensor(r, dtype=torch.float32, device=device) for r in rewards]
    print(f"Recompnsa: {rewards[0]}")

    print(f"Para ppo_trainer.step:")
    print(f"  len(queries_for_step): {len(queries_for_step)}, shape del primer elemento: {queries_for_step[0].shape if queries_for_step else 'N/A'}")
    print(f"  len(responses_generated_only_for_step): {len(responses_generated_only_for_step)}, shape del primer elemento: {responses_generated_only_for_step[0].shape if responses_generated_only_for_step else 'N/A'}")
    print(f"  len(rewards_pt): {len(rewards_pt)}, shape del primer elemento: {rewards_pt[0].shape if rewards_pt else 'N/A'}")

    stats = ppo_trainer.step(queries_for_step, responses_generated_only_for_step, rewards_pt)
    print(f"Stats del paso PPO: {stats}")

print("\n--- Entrenamiento PPO Completado ---")

# ... (generación post-entrenamiento) ... (evaluacion para ver si aprendió)
print("\nGenerando ejemplos después del entrenamiento con el modelo PPO:")
for prompt_text_val in prompts_text:
    inputs = tokenizer(prompt_text_val, return_tensors="pt").to(device)
    gen_kwargs_val = generation_config.to_dict()
    if "attention_mask" in inputs:
        gen_kwargs_val["attention_mask"] = inputs["attention_mask"]

    response_ids = ppo_trainer.model.generate(
        input_ids=inputs["input_ids"],
        **gen_kwargs_val
    )
    prompt_length = inputs.input_ids.shape[1]
    generated_tokens = response_ids[0][prompt_length:]
    response_generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    print(f"Prompt: {prompt_text_val}")
    print(f"Respuesta generada: {response_generated_text}")
    print("-" * 30)

Usando dispositivo: cuda


Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Iniciando entrenamiento PPO para 2 pasos...

--- PPO Step 1/2 ---
Query (texto): Say something friendly:
  ADVERTENCIA: ppo_trainer.generate devolvió 2D, se hizo squeeze(0). Nueva forma: torch.Size([14])
Respuesta (solo generada):  Visit this page to enter your email address.
Recompnsa: -1.0
Para ppo_trainer.step:
  len(queries_for_step): 1, shape del primer elemento: torch.Size([4])
  len(responses_generated_only_for_step): 1, shape del primer elemento: torch.Size([10])
  len(rewards_pt): 1, shape del primer elemento: torch.Size([])


Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Stats del paso PPO: {'objective/kl': 0.0, 'objective/kl_dist': 0.0, 'objective/logprobs': array([[-7.8799977 , -9.27855   , -5.05488   , -8.725997  , -3.7739468 ,
        -2.3050778 , -1.5005264 , -5.639865  , -1.1585556 , -0.5788778 ,
        -0.07093947, -0.5797623 , -1.4295491 ]], dtype=float32), 'objective/ref_logprobs': array([[-7.8799977 , -9.27855   , -5.05488   , -8.725997  , -3.7739468 ,
        -2.3050778 , -1.5005264 , -5.639865  , -1.1585556 , -0.5788778 ,
        -0.07093947, -0.5797623 , -1.4295491 ]], dtype=float32), 'objective/kl_coef': 0.2, 'objective/entropy': 25.76309585571289, 'ppo/mean_non_score_reward': 0.0, 'ppo/mean_scores': -1.0, 'ppo/std_scores': nan, 'tokens/queries_len_mean': 4.0, 'tokens/queries_len_std': nan, 'tokens/queries_dist': 4.0, 'tokens/responses_len_mean': 10.0, 'tokens/responses_len_std': nan, 'tokens/responses_dist': 10.0, 'ppo/loss/policy': -0.10691343247890472, 'ppo/loss/value': 29.548873901367188, 'ppo/loss/total': 2.8479743003845215, 'ppo/po

Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Stats del paso PPO: {'objective/kl': 0.03426489233970642, 'objective/kl_dist': 0.03426489233970642, 'objective/logprobs': array([[ -7.807659 ,  -4.2594423,  -4.644397 ,  -5.597473 ,  -0.5195322,
         -4.4423513,  -2.4102228,  -5.4650035,  -2.0412884,  -0.8798482,
         -2.6848269,  -3.0010023,  -4.8528886,  -3.650228 ,  -1.1317309,
         -8.083513 , -10.920279 ,  -1.2821956,  -1.9280808,  -7.1827497,
         -6.609927 ,  -5.5634413,  -4.9190083,  -2.3282356,  -1.2853012,
         -4.2008743]], dtype=float32), 'objective/ref_logprobs': array([[ -7.850729  ,  -4.19932   ,  -4.668183  ,  -5.254793  ,
         -0.48892888,  -4.336046  ,  -2.3925164 ,  -5.5573225 ,
         -1.9978671 ,  -0.87309617,  -2.8433628 ,  -2.7482572 ,
         -4.960182  ,  -3.953886  ,  -1.2977333 ,  -8.2495575 ,
        -11.046376  ,  -1.2424303 ,  -2.0612853 ,  -7.071454  ,
         -6.612035  ,  -5.560278  ,  -4.8230877 ,  -2.2695513 ,
         -1.28824   ,  -4.08598   ]], dtype=float32), 'objective

Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Prompt: Say something friendly:
Respuesta generada:  "Hey you're making a toast?"

Hey: I'm making a toast, dude.
What: I went to dinner with.

------------------------------
Prompt: Introduce yourself:
Respuesta generada: 

"Would you like to know if I can make it to our house?".

"Would you like to know if I can please"?
------------------------------
